# Role-Based Access Control (RBAC) Decorator Factory

You are a DevOps engineer building an internal admin tool. Several functions in this tool perform sensitive operations, and access must be restricted based on user roles. To avoid repeating permission checks, you will build a configurable decorator. For this first part, you will focus on the core logic of the decorator factory, assuming the inputs will always be correctly formatted.

Your task is to create a decorator factory named `require_role`. This factory will generate a decorator that checks if a user has the specified role.

## Functional Requirements

- `require_role` must be a decorator factory that accepts one argument: `required_role` (a string like `'admin'`).

- The decorator it generates must wrap a function.

- The wrapper must assume it will always receive a keyword argument named `user`.

- The `user` argument will be a dictionary with a `roles` key, which is a list of strings (for example, `{'name': 'alice', 'roles': ['admin', 'viewer']}`).

- If the user's `roles` list contains the `required_role`, the original function should be executed.

- If the user does not have the `required_role`, the decorator must raise a `PermissionError`.

- The decorator must correctly return whatever value the original function returns.

- The decorator must preserve the original function's metadata (`__name__`, `__doc__`).

## Important Note

You do not need to validate the inputs in this part. You can assume `user` will always be provided as a keyword argument and will always be a dictionary with a `roles` list. We will add robust validation in another exercise after we cover exception handling in Python!

## Hint

Even though we haven't covered exception handling yet, here is how you can raise a `PermissionError` exception from within an `if` statement:

```python
if <some condition>:
    raise PermissionError("some informative message")
```

This will ensure that the exception will be raised only when the condition evaluates to `True`.

## Example

```python
@require_role('admin')
def restart_server(*, user, server_id):
    """Restarts a server after a permission check."""
    return f"Server {server_id} restart initiated by {user['name']}."

admin_user = {'name': 'alice', 'roles': ['admin', 'viewer']}
viewer_user = {'name': 'bob', 'roles': ['viewer']}

# This call will succeed
restart_server(user=admin_user, server_id='web-01')

# This call will raise a PermissionError
# restart_server(user=viewer_user, server_id='db-01')
```

## How Your Solution Will Be Tested

- A successful call where the user has the required role.
- A failed call where the user lacks the required role (must raise `PermissionError`).
- Preservation of the wrapped function's metadata and return value.

In [1]:
from functools import wraps
def require_role(required_role):
    """
    A decorator factory that creates a decorator to check for a specific user role.

    Args:
        required_role (str): The role string that the user must have.

    Returns:
        A decorator function.
    """

    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            if required_role in kwargs["user"]["roles"]:
                return func(*args, **kwargs)
            else:
                raise PermissionError(f"User does not have the required role: {required_role}")
        return wrapper
    return decorator

@require_role('admin')
def restart_server(*, user, server_id):
    """Restarts a server after a permission check."""
    return f"Server {server_id} restart initiated by {user['name']}."

admin_user = {'name': 'alice', 'roles': ['admin', 'viewer']}
viewer_user = {'name': 'bob', 'roles': ['viewer']}

restart_server(user=admin_user, server_id='web-01')
#restart_server(user=viewer_user, server_id='db-01')

'Server web-01 restart initiated by alice.'